In [ ]:
"""PP-GPAT post-processor test script for the GPAT model."""

import os

import matplotlib.pyplot as plt
import pandas as pd

from pycontrails.models.gpat.pp_gpat import GPATPostProcessor

In [ ]:
outputs_dir = f"{os.getcwd()}/outputs/"

In [ ]:
# Filter criteria
criteria = {
    # "n_ac": 3,
    # "rt_fl": (pd.Timedelta(minutes=30), pd.Timedelta(hours=2)),
    # "date_created": (pd.Timestamp("2024-11-16"), pd.Timestamp("2024-11-17")),
    "job_id": "bg_run_NA_2022-01-01T12_00_00"
}

In [ ]:
pp_gpat = GPATPostProcessor(outputs_dir, criteria)
pp_gpat.filtered_df

In [ ]:
# Create dicts to hold all necessary data (but no more)
fl_df_dict = {}
pl_df_dict = {}
chem_ds_dict = {}

In [ ]:
for job_id in pp_gpat.job_ids:
    # fl_df_dict[job_id] = pp_gpat.load_fl_df(job_id)
    # pl_df_dict[job_id] = pp_gpat.load_pl_df(job_id)
    chem_ds_dict[job_id] = pp_gpat.load_chem_ds(job_id, 11, 11, 2)

In [ ]:
# bg run plots
fig, ax = plt.subplots(2, 4, figsize=(20, 10))

for _job_id, chem_ds in chem_ds_dict.items():
    print(chem_ds.variables)
    # get time var
    time = chem_ds["time"].values

    # get month from time var
    month = pd.to_datetime(time).month

    # species to plot
    NO = chem_ds["Y"].sel(species_out="NO").values
    NO2 = chem_ds["Y"].sel(species_out="NO2").values
    NOx = NO + NO2
    O3 = chem_ds["Y"].sel(species_out="O3").values
    OH = chem_ds["Y"].sel(species_out="OH").values
    HO2 = chem_ds["Y"].sel(species_out="HO2").values
    CO = chem_ds["Y"].sel(species_out="CO").values
    CH4 = chem_ds["Y"].sel(species_out="CH4").values

    NOy = pp_gpat.calc_NOy(chem_ds)
    NOz = pp_gpat.calc_NOz(chem_ds)

    NO2t = pp_gpat.calc_NO2t(chem_ds)
    O3_NOz = pp_gpat.calc_O3_NOz(chem_ds, NOz)
    HCHO_NO2 = pp_gpat.calc_HCHO_NO2(chem_ds)
    H2O2_HNO3 = pp_gpat.calc_H2O2_HNO3(chem_ds)
    alpha_CH3O2 = pp_gpat.calc_alpha_CH3O2(chem_ds)

In [ ]:
ax[0, 0].plot(time, NO)
ax[0, 1].plot(time, NO2)
ax[0, 2].plot(time, O3)
ax[0, 3].plot(time, NOy)
ax[1, 0].plot(time, OH)
ax[1, 1].plot(time, HO2)
ax[1, 2].plot(time, CO)
ax[1, 3].plot(time, CH4)
# pp_gpat.plot_line_plot(ax[0, 1], time, NO2, 'NO2', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[0, 2], time, O3, 'O3', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[0, 3], time, NOy, 'NOy', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 0], time, OH, 'OH', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 1], time, HO2, 'HO2', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 2], time, CO, 'CO', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 3], time, CH4, 'CH4', 'Time', 'Concentration (ppb)')

fig.suptitle("Time series of species concentrations for bg run NA Jan")

plt.show()